# Structured Extensions (V2)

This notebook is the V2 companion of `../../extension-attributes/extension_attributes.ipynb`. It shows how to store your own data in the `extension` of test results, steps, and measurements.

In V2, an extension is a JSON object. It can hold nested objects and lists, not just text.

**Note:** V2 does not accept a `schema_id` on test results, steps, or measurements yet, so the service does not check extensions against a schema. Numbers are stored as JSON numbers, so whole numbers are read back as `float` values (for example, `3` is read back as `3.0`).

## Prerequisites

- NI Measurement Data Services running in V2 mode
- Python environment with the `ni.datastore` package (see `../README.md`)

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient

data_store_client = DataStoreClient(grpc_channel=grpc_channel)

## Create Data with Extensions

In [ ]:
from ni.datastore.data.v2 import Step, TestResult

test_result_id = data_store_client.create_test_result(
    TestResult(
        name="Structured Extensions Example",
        extension={
            "fixture": {"id": "FIX-12", "slots": [1, 2, 3]},
            "retest": False,
            "operator_notes": "Second run after recalibration.",
        },
    )
)
step_id = data_store_client.create_step(
    Step(
        name="Warm-up",
        test_result_id=test_result_id,
        extension={"phase": {"name": "warmup", "seconds": 30}},
    )
)
measurement_id = data_store_client.create_measurement(
    "Output Voltage",
    5.01,
    step_id,
    units="V",
    extension={"probe": {"channel": 3, "attenuation": "10x"}},
)
print(f"Created Test Result: {test_result_id}")

## Read Extensions

In [ ]:
test_result = data_store_client.get_test_result(test_result_id)
step = data_store_client.get_step(step_id)
measurement = data_store_client.get_measurement(measurement_id)

print(f"test result fixture: {test_result.extension['fixture']}")
print(f"test result retest: {test_result.extension['retest']}")
print(f"step phase: {step.extension['phase']}")
print(f"measurement probe: {measurement.extension['probe']}")

assert test_result.extension["fixture"]["id"] == "FIX-12"
assert test_result.extension["fixture"]["slots"] == [1, 2, 3]
assert measurement.extension["probe"]["channel"] == 3

## Update Extensions

Updating `extension` replaces the whole extension, so include every key that you want to keep.

In [ ]:
new_extension = dict(test_result.extension)
new_extension["retest"] = True
new_extension["fixture"] = {"id": "FIX-13", "slots": [1, 2]}

updated_test_result = data_store_client.update_test_result(test_result_id, extension=new_extension)
print(f"updated extension: {updated_test_result.extension}")
assert updated_test_result.extension["retest"] is True
assert updated_test_result.extension["operator_notes"] == "Second run after recalibration."

## Clean up

In [ ]:
data_store_client.delete_test_result(test_result_id, force=True)
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()